# exp112 learned PF likelihood weight / feature follow-up

## Contents

1. Setup and configuration
2. Input artifact checks
3. Run posthoc audit
4. Metrics and artifacts preview

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, get_nested, load_config
from learned_pf_likelihood_weight_or_feature_followup import run_followup

paths = ExperimentPaths()
config = load_config()
output_dir = Path('/kaggle/working/artifacts') if Path('/kaggle/working').exists() else paths.artifacts_dir
output_dir.mkdir(parents=True, exist_ok=True)

print('experiment:', get_nested(config, 'experiment.name'))
print('route:', get_nested(config, 'experiment.route'))
print('parent:', get_nested(config, 'lineage.parent'))
print('mode:', get_nested(config, 'followup.mode'))
print('output_dir:', output_dir)

## 2. Input artifact checks

In [ ]:
exp111_dir = Path(get_nested(config, 'data.exp111_artifact_dir_local'))
exp111_long_name = get_nested(config, 'data.exp111_oof_likelihood_long')
exp099_wide = Path(get_nested(config, 'data.exp099_train_feature_cache_local'))

print('exp111 artifact dir:', exp111_dir)
print('exp111 long cache:', exp111_long_name)
print('exp099 wide cache:', exp099_wide)
print('local exp111 exists:', (exp111_dir / exp111_long_name).exists())
print('local exp099 exists:', exp099_wide.exists())
print('candidates:', get_nested(config, 'followup.candidates'))
print('alphas:', get_nested(config, 'followup.pf_weight_alphas'))

## 3. Run posthoc audit

In [ ]:
summary = run_followup(
    output_dir=output_dir,
    exp111_artifact_dir=get_nested(config, 'data.exp111_artifact_dir_local'),
    exp111_long_path=get_nested(config, 'data.exp111_oof_likelihood_long'),
    exp099_wide_path=get_nested(config, 'data.exp099_train_feature_cache_local'),
    max_groups=get_nested(config, 'followup.max_groups'),
)
print(json.dumps(summary['decision'], indent=2, sort_keys=True))

## 4. Metrics and artifacts preview

In [ ]:
metrics_path = output_dir / 'exp112_learned_pf_likelihood_weight_or_feature_followup_metrics.csv'
feature_schema_path = output_dir / 'exp112_learned_pf_likelihood_weight_or_feature_followup_feature_schema.csv'
summary_path = output_dir / 'exp112_learned_pf_likelihood_weight_or_feature_followup_summary.json'

metrics = pd.read_csv(metrics_path)
display(metrics.sort_values(['mode', 'rmse_tvt']).head(20))

feature_schema = pd.read_csv(feature_schema_path)
display(feature_schema.head(40))

with summary_path.open() as fp:
    saved_summary = json.load(fp)
print('rows:', saved_summary['rows'])
print('wells:', saved_summary['wells'])
print('artifacts:', saved_summary['artifacts'])
print('sha256:', saved_summary['sha256'])